# OpenLearn-AI — Week 6 AI Pipeline Demonstration

Real end-to-end execution of the pipeline **as actually implemented** in Week 6:

```
PDF → Docling → targeted OCR decision → Gemini OCR (when required) → CanonicalDocument
    → deterministic chunking → BGE-M3 embeddings → PostgreSQL/pgvector → similarity search
```

Every stage below calls the **real production implementations** in `backend/app`.
No mocks are used for the main execution.

**Prerequisites to run this notebook**

* Kernel: the backend virtual environment (`backend/.venv`) so `app.*` imports resolve.
* `backend/.env` contains `GEMINI_API_KEY=...` (never displayed by this notebook).
* The development PostgreSQL/pgvector service is running (`docker compose -p infra up -d`)
  and migrations are applied (`alembic upgrade head` → `f3a1b7c9d4e2`).
* First BGE-M3 run downloads the model to the local HF cache (subsequent runs reuse it).
* The default input is a real scanned page from the OCR benchmark corpus.

## What this notebook demonstrates

* Docling ingestion with **built-in PDF OCR intentionally disabled** (`PdfPipelineOptions(do_ocr=False)`).
* The application-owned **targeted OCR decision** (`needs_ocr`) — OCR only where the extracted text is too short.
* A **real Gemini API call** (`gemini-2.5-flash`) through the PAL OCR interface for eligible pages.
* The normalized `CanonicalDocument` produced by ingestion and enriched by OCR.
* **Deterministic structure-aware chunking** with `chunk_id = "{document_id}:{seq}"`.
* **Real BGE-M3 embeddings** (1024-dimensional, L2-normalized, lazy model load).
* **Real PostgreSQL + pgvector persistence** via the existing `VectorDBInterface` provider.
* A **real cosine similarity search** answered from the database.

**Out of scope (not demonstrated, not claimed):** advanced RAG, reasoning orchestration,
streaming, WebSockets, background ingestion jobs. Interfaces for some of these exist in PAL,
but they are not part of this demonstration.

## Architecture overview

```
PDF (scanned or born-digital)
        |
        v
Docling ingestion  ---  PdfPipelineOptions(do_ocr=False)   [built-in OCR OFF]
        |
        v
CanonicalDocument (pages + extracted text)
        |
        v
needs_ocr(page.text)                 [application decision, ocr_min_text_chars]
        |
        +-- not required -------------------------------+
        |                                              |
        v                                              |
Gemini OCR via PAL (REAL API)                          |
  source: real single-page PDF artifact (pypdfium2)    |
        |                                              |
        v                                              |
CanonicalDocument (OCR text + ocr metadata)            |
        |                                              |
        v                                              v
Deterministic chunking   chunk_id = '{document_id}:{seq}'
        |
        v
BGE-M3 embeddings   (1024-d, L2-normalized, device auto: CUDA local / CPU AWS)
        |
        v
PostgreSQL + pgvector   (vector_records: VECTOR(1024) + JSONB metadata)
        |
        v
Similarity search   (cosine, score = 1 - cosine_distance)
```

Current project decisions reflected above: Gemini is the application OCR provider;
RapidOCR remains available as a local Docling capability (untouched); PaddleOCR is not
the application OCR path; OpenAI embedding fallback is deferred and not implemented;
PAL provides the provider abstraction — there is no plugin/discovery/orchestration layer.

In [ ]:
import shutil
import sys
import tempfile
import time
from pathlib import Path

# The notebook lives at the repository root; make backend/ importable (app.*).
REPO_ROOT = Path.cwd() if (Path.cwd() / "backend").exists() else Path.cwd().parent
BACKEND_DIR = REPO_ROOT / "backend"
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

# ---------------------------------------------------------------------------
# INPUT DOCUMENT — change this single variable to demo a different document.
# ---------------------------------------------------------------------------
PDF_PATH = Path(
    REPO_ROOT,
    "experiments/OCR/ocr-benchmark/data/raw/custom",
    "3.English scanned",
    "custom_custom_english_scanned_003_p001.pdf",
)

# Alternative corpus examples — adjust folder/file names to documents that exist
# in your local OCR benchmark corpus (same data/raw/custom tree):
# PDF_PATH = REPO_ROOT / "experiments/OCR/ocr-benchmark/data/raw/custom/<english-born-digital>/<file>.pdf"
# PDF_PATH = REPO_ROOT / "experiments/OCR/ocr-benchmark/data/raw/custom/<arabic-born-digital>/<file>.pdf"
# PDF_PATH = REPO_ROOT / "experiments/OCR/ocr-benchmark/data/raw/custom/<arabic-scanned>/<file>.pdf"
# PDF_PATH = REPO_ROOT / "experiments/OCR/ocr-benchmark/data/raw/custom/<mixed-arabic-english>/<file>.pdf"

# Notebook-local demo settings (composition only — NOT production configuration):
DEMO_DOCUMENT_PREFIX = "week6-demo"   # temporary pgvector document_id (deterministic -> idempotent re-runs)
OCR_ARTIFACT_DIR = Path(tempfile.mkdtemp(prefix="week6-demo-ocr-artifacts-"))

# Semantic search query — tune it to the selected document's topic:
SEARCH_QUERY = "What is the main topic of this document?"

TIMINGS = {}
PIPELINE_START = time.perf_counter()

In [ ]:
import time

import pandas as pd

# --- Real production components (imported, not re-implemented) ---
from app.config import settings
from app.documents.chunking import chunk_document
from app.pal.factory import get_embedding_provider, get_ocr_provider, get_vector_db_provider
from app.pal.models.types import VectorRecord
from app.services.ingestion import ingest_document
from app.services.ocr import enrich_document_with_ocr, needs_ocr
from app.services.ocr_source import pdf_page_source_resolver


def preview(text: str, width: int = 90) -> str:
    """Notebook-local helper: single-line, width-capped text preview."""
    flat = " ".join((text or "").split())
    return flat[:width] + "…" if len(flat) > width else flat


# --- Fail fast, with clear guidance and zero secret exposure ---
if not PDF_PATH.exists():
    raise FileNotFoundError(f"Input document not found: {PDF_PATH}")
if not settings.gemini_api_key:
    raise RuntimeError(
        "GEMINI_API_KEY is not configured. Add GEMINI_API_KEY=<your key> to "
        "backend/.env and restart the kernel. (The value is never displayed.)"
    )

pd.DataFrame(
    [
        ("input document", PDF_PATH.name),
        ("OCR provider", "gemini (real API)"),
        ("OCR model", settings.ai_ocr_model),
        ("OCR threshold (ocr_min_text_chars)", settings.ocr_min_text_chars),
        ("Embedding provider", "bge-m3 (real local model)"),
        ("Embedding model", settings.ai_embedding_model),
        ("Embedding dimension", settings.ai_embedding_dimension),
        ("Embedding device", settings.ai_embedding_device),
        ("Chunk size / overlap", f"{settings.chunk_size} / {settings.chunk_overlap}"),
        ("Vector store", f"PostgreSQL + pgvector @ {settings.database_url.split('@')[-1]}"),
    ],
    columns=["setting", "value"],
)

## Input document

The default input is a **real scanned page** from the OCR benchmark corpus — exactly the
kind of document the targeted OCR path was built for.

In [ ]:
PDF_PATH, f"{PDF_PATH.stat().st_size / 1024:.1f} KB"

## Stage 1 — Docling conversion

`app.services.ingestion.ingest_document` runs the real Docling converter. For PDFs it
constructs `PdfPipelineOptions(do_ocr=False)` — Docling's built-in OCR stage is
**intentionally disabled** so that scanned pages come out with little/no text and are
routed to the application's targeted OCR decision in the next stage.

Docling conversion and application OCR are deliberately separate responsibilities.

In [ ]:
t0 = time.perf_counter()
document = ingest_document(PDF_PATH)          # real Docling ingestion
docling_seconds = time.perf_counter() - t0
TIMINGS["1. Docling ingestion"] = round(docling_seconds, 2)

display(
    pd.DataFrame(
        [
            {
                "page": p.page_number,
                "chars": len(p.text),
                "preview": preview(p.text, 70),
            }
            for p in document.pages
        ]
    )
)

{
    "document_id (from file stem)": document.document_id,
    "conversion_status": document.metadata.get("conversion_status"),
    "page_count": document.metadata.get("page_count"),
    "total_chars": sum(len(p.text) for p in document.pages),
    "docling_builtin_pdf_ocr": "DISABLED — PdfPipelineOptions(do_ocr=False) in app/services/ingestion.py",
}

## Stage 2 — Targeted OCR decision

The application decides — per page — whether OCR is needed, using the existing
`needs_ocr(page.text)` gate: OCR is required when the extracted text is shorter than
`ocr_min_text_chars` (exactly at the threshold is **not** OCR). This is why the scanned
page below is routed to Gemini while text-rich pages would never reach the OCR provider.

In [ ]:
threshold = settings.ocr_min_text_chars
plan = [
    {
        "page": p.page_number,
        "chars": len(p.text),
        "threshold": threshold,
        "ocr_required": needs_ocr(p.text),
        "current_text_preview": preview(p.text, 55),
    }
    for p in document.pages
]
display(pd.DataFrame(plan))

PAGES_FOR_OCR = [row["page"] for row in plan if row["ocr_required"]]
f"Pages routed to Gemini OCR: {PAGES_FOR_OCR or 'none (document text is sufficient)'}"

## Stage 3 — Real Gemini OCR

Eligible pages flow through the production orchestration
(`enrich_document_with_ocr`) using:

* the **real Gemini provider** from the factory (`get_ocr_provider("gemini")`, model
  `gemini-2.5-flash` from settings) — this makes a **real Gemini API request**;
* the **real single-page PDF artifact resolver** (`pdf_page_source_resolver`), which
  hands the provider a real one-page PDF (written to a notebook-local temp dir, not
  into the repository).

The key comes from `backend/.env` via the existing `Settings` and is never printed.
If Gemini fails, this stage fails visibly — the notebook does not simulate success.
If no page met the threshold, Gemini is not called at all (and this stage says so).

In [ ]:
gemini_ocr = get_ocr_provider("gemini")                            # REAL provider (Gemini API)
resolver = pdf_page_source_resolver(output_dir=OCR_ARTIFACT_DIR)   # REAL single-page artifacts

# Notebook-local observer: delegates every call to the REAL provider unchanged and
# only records timing/result info for the demo table. Not a mock, not a second
# implementation of the provider.
ocr_calls: list[dict] = []


class _ObservedOCR:
    provider_name = gemini_ocr.provider_name

    async def health_check(self):
        return await gemini_ocr.health_check()

    async def extract_text(self, source: str):
        started = time.perf_counter()
        result = await gemini_ocr.extract_text(source)   # <-- REAL Gemini request
        ocr_calls.append(
            {
                "provider": result.provider,
                "model": result.metadata.get("model"),
                "artifact": Path(source).name,
                "result_chars": len(result.text),
                "finish_reason": result.metadata.get("finish_reason"),
                "seconds": round(time.perf_counter() - started, 2),
                "text": result.text,
            }
        )
        return result

    async def extract_text_batch(self, sources):
        return [await self.extract_text(s) for s in sources]


t0 = time.perf_counter()
document = await enrich_document_with_ocr(document, _ObservedOCR(), resolver)
ocr_seconds = time.perf_counter() - t0
TIMINGS["2+3. Targeted OCR (decision + Gemini)"] = round(ocr_seconds, 2)

if not ocr_calls:
    print("No page met the OCR threshold — Gemini was NOT called for this document.")
else:
    display(pd.DataFrame([{k: v for k, v in c.items() if k != "text"} for c in ocr_calls]))
    for call in ocr_calls:
        print(f"\n--- Gemini OCR preview · {call['artifact']} ---")
        print(preview(call["text"], 260))

## Stage 4 — CanonicalDocument after targeted OCR

The OCR output has become part of the normalized document: page text was replaced by the
Gemini result and each OCR-ed page carries an `ocr` metadata block (`applied`, `provider`,
`source`, provider metadata).

One notebook-local composition step: the stored vectors are scoped to a **clearly
identifiable temporary `document_id`** (`week6-demo--<original-stem>`). It is
deterministic, so re-running the notebook overwrites the same rows (idempotent upsert)
and cleanup removes exactly this document — nothing else.

In [ ]:
DEMO_DOCUMENT_ID = f"{DEMO_DOCUMENT_PREFIX}--{document.document_id}"
document.document_id = DEMO_DOCUMENT_ID

display(
    pd.DataFrame(
        [
            {
                "page": p.page_number,
                "chars_after_ocr": len(p.text),
                "ocr_applied": p.metadata.get("ocr", {}).get("applied", False),
                "ocr_provider": p.metadata.get("ocr", {}).get("provider"),
                "preview": preview(p.text, 60),
            }
            for p in document.pages
        ]
    )
)

{
    "document_id (demo-scoped)": document.document_id,
    "source": document.source,
    "title": document.title,
    "pages": [p.page_number for p in document.pages],
    "language": document.language,
}

## Stage 5 — Deterministic chunking

The application-owned structure-aware chunker (`app.documents.chunking.chunk_document`)
splits paragraph-first, keeps page provenance, and assigns IDs as
`{document_id}:{seq}` (zero-based). Re-chunking the same document yields identical IDs
and ordering — asserted below, not just claimed.

In [ ]:
t0 = time.perf_counter()
chunks = chunk_document(document)            # real deterministic chunker (P8)
TIMINGS["5. Deterministic chunking"] = round(time.perf_counter() - t0, 4)

display(
    pd.DataFrame(
        [
            {
                "chunk_id": c.chunk_id,
                "seq": c.chunk_id.rsplit(":", 1)[-1],
                "pages": c.pages,
                "chars": c.metadata.get("char_count"),
                "preview": preview(c.text, 60),
            }
            for c in chunks
        ]
    )
)

# Determinism proof: same document -> identical chunk IDs and ordering.
assert [c.chunk_id for c in chunks] == [c.chunk_id for c in chunk_document(document)]
f"chunk_id rule '{{document_id}}:{{seq}}' verified — {len(chunks)} deterministic chunks"

## Stage 6 — BGE-M3 embeddings

The real `BGEM3EmbeddingProvider` from the factory embeds all chunks in one batch.
The model is **lazy-loaded on the first call** (this cell's timing therefore includes
the one-time load on a fresh kernel; later calls reuse the cached model). Vectors are
1024-dimensional and L2-normalized (norms ≈ 1.0), with device selected by
`ai_embedding_device=auto` (CUDA locally, CPU on the GPU-less AWS server).

In [ ]:
embedding_provider = get_embedding_provider("bge-m3")    # REAL BGE-M3 (lazy-loaded)

t0 = time.perf_counter()
embeddings = await embedding_provider.embed_batch([c.text for c in chunks])
TIMINGS["6. BGE-M3 embeddings (first call loads the model)"] = round(time.perf_counter() - t0, 2)

norms = [sum(v * v for v in e.vector) ** 0.5 for e in embeddings]
{
    "provider": embeddings[0].provider,
    "model": embeddings[0].model,
    "device": embeddings[0].metadata.get("device"),
    "normalized": embeddings[0].metadata.get("normalized"),
    "chunks_embedded": len(embeddings),
    "dimension (provider.dimension)": embedding_provider.dimension,
    "vector_length": len(embeddings[0].vector),
    "norms min..max": [round(min(norms), 6), round(max(norms), 6)],
    "vector[0][:5] preview": [round(v, 5) for v in embeddings[0].vector[:5]],
}

## Stage 7 — PostgreSQL / pgvector persistence

The real `PostgresVectorDBProvider` (via the factory, bound to the application's
`AsyncSession`) stores each chunk as a `VectorRecord` — embedding `VECTOR(1024)`,
content, and JSONB provenance metadata (`document_id`, `pages`, `language`,
`section`, `title`). The provider **never commits**; the notebook, as the caller,
owns the transaction. Only this demo document's rows are written.

In [ ]:
from app.db.session import AsyncSessionLocal

session = AsyncSessionLocal()
vector_db = get_vector_db_provider("postgres", session=session)   # REAL pgvector provider

health = await vector_db.health_check()
if not health.healthy:
    raise RuntimeError(
        f"PostgreSQL is not reachable ({health.message}). Start the compose "
        "database service (e.g. `docker compose -p infra up -d`) and re-run."
    )

records = [
    VectorRecord(
        id=chunk.chunk_id,
        vector=embedding.vector,
        content=chunk.text,
        metadata={
            "document_id": chunk.document_id,
            "pages": chunk.pages,
            "language": chunk.language,
            "section": chunk.section,
            "title": document.title,
        },
    )
    for chunk, embedding in zip(chunks, embeddings, strict=True)
]

t0 = time.perf_counter()
await vector_db.upsert(records)
await session.commit()      # caller owns the transaction (provider never commits)
TIMINGS["7. pgvector persistence (upsert + commit)"] = round(time.perf_counter() - t0, 3)

{
    "records_upserted": len(records),
    "vector_dimension": len(records[0].vector),
    "document_id": DEMO_DOCUMENT_ID,
    "record_ids": [r.id for r in records],
    "metadata_keys": sorted(records[0].metadata.keys()),
}

## Stage 8 — Real vector similarity search

The query is embedded with the **same real BGE-M3 model** and searched against
pgvector through the existing provider (cosine distance, `score = 1 - distance`,
filtered to this demo document). The rows below are read back from PostgreSQL —
nothing is fabricated.

> Tip: make `SEARCH_QUERY` (configuration cell) specific to the selected document so
> the ranking is meaningful for the demo.

In [ ]:
t0 = time.perf_counter()
query_embedding = await embedding_provider.embed(SEARCH_QUERY)   # REAL BGE-M3 query vector
results = await vector_db.search(
    query_embedding.vector,
    top_k=5,
    filters={"document_id": DEMO_DOCUMENT_ID},
)
TIMINGS["8. Query embedding + pgvector similarity search"] = round(time.perf_counter() - t0, 3)

display(
    pd.DataFrame(
        [
            {
                "rank": i + 1,
                "similarity": round(r.score, 4),
                "chunk_id": r.id,
                "pages": r.metadata.get("pages"),
                "preview": preview(r.content or "", 80),
            }
            for i, r in enumerate(results)
        ]
    )
)
f"'{SEARCH_QUERY}' — {len(results)} result(s) read back from PostgreSQL (score = 1 - cosine_distance)"

## End-to-end timing

Wall-clock times measured with `time.perf_counter()` around each real stage
(standard library only). Stage 6 includes the one-time lazy model load; the OCR stage
includes real Gemini API latency.

In [ ]:
TIMINGS["TOTAL (Stage 1 -> 8)"] = round(time.perf_counter() - PIPELINE_START, 2)
pd.DataFrame([{"stage": k, "seconds": v} for k, v in TIMINGS.items()])

## Implementation status

| Component | Demonstrated | Execution |
|---|---|---|
| Docling conversion (built-in PDF OCR disabled) | Yes | Real |
| Targeted OCR decision (`needs_ocr`) | Yes | Real |
| Single-page PDF artifact resolver | Yes | Real (pypdfium2) |
| Gemini OCR | Yes | Real API |
| CanonicalDocument enrichment | Yes | Real |
| Deterministic chunking (`{document_id}:{seq}`) | Yes | Real |
| BGE-M3 embeddings (1024-d, normalized) | Yes | Real model |
| PostgreSQL/pgvector persistence | Yes | Real database |
| Vector similarity search | Yes | Real database |
| Demo-scoped cleanup | Yes | Real database |

**Explicitly outside this notebook's scope** (not demonstrated, not claimed complete):
advanced RAG, reasoning orchestration, streaming responses, WebSockets chat,
background ingestion jobs, OCR fallback chains, OpenAI embedding fallback (deferred).

## Cleanup

Removes exactly what this notebook created — nothing else:

* the demo's `vector_records` rows, deleted via the existing delete-by-metadata-filter API
  (`filters={"document_id": DEMO_DOCUMENT_ID}`);
* the temporary single-page OCR artifacts (system temp dir).

Cleanup failures are reported, never hidden. Because the demo `document_id` is
deterministic, a later re-run overwrites the same rows, so an interrupted run cannot
accumulate duplicates.

In [ ]:
try:
    deleted = await vector_db.delete(filters={"document_id": DEMO_DOCUMENT_ID})
    await session.commit()
    shutil.rmtree(OCR_ARTIFACT_DIR, ignore_errors=True)
    await session.close()
    cleanup_result = {"status": "ok", "deleted_vector_records": deleted}
except Exception as exc:
    cleanup_result = {
        "status": "CLEANUP_FAILED — reported, not hidden",
        "error": repr(exc),
        "demo_document_id": DEMO_DOCUMENT_ID,
        "note": "Re-run this cell; re-running the demo overwrites the same deterministic IDs.",
    }
cleanup_result

---

**To try another document:** change `PDF_PATH` in the configuration cell (commented
corpus examples are provided there), adjust `SEARCH_QUERY` to its topic, and
**Restart & Run All**. Born-digital documents with sufficient text will visibly skip
the Gemini stage; scanned pages will visibly exercise it.

This notebook is a demonstration composition layer over the existing Week 6
implementation — it introduces no new production architecture.